In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp04/ex_3.py ──
"""
Shared infrastructure for MLFP04 Exercise 3 — Dimensionality Reduction.

Contains: data loading, scaling, common output directory, KMeans-based
silhouette evaluation in the embedding space. Technique-specific code
(PCA/KPCA/t-SNE/UMAP algorithms and their plots) lives in the per-
technique files, NOT here.

"""

from pathlib import Path

import numpy as np
import polars as pl
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# OUTPUT + REPRODUCIBILITY
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "ex3_dimreduce"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
DEFAULT_N_CLUSTERS = 4

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — E-commerce customers (reused from MLFP03)
# ════════════════════════════════════════════════════════════════════════


def load_customer_matrix() -> tuple[np.ndarray, list[str], pl.DataFrame]:
    """Load e-commerce customers, standardise numeric features.

    Returns:
        X          : (n_samples, n_features) standardised float matrix
        feature_cols: list of feature column names in order
        df_raw     : the raw polars DataFrame before scaling
    """
    loader = MLFPDataLoader()
    customers = loader.load("mlfp03", "ecommerce_customers.parquet")

    feature_cols = [
        c
        for c, d in zip(customers.columns, customers.dtypes)
        if d in (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
        and c not in ("customer_id",)
    ]

    df_clean = customers.drop_nulls(subset=feature_cols)
    X_raw, _, _ = to_sklearn_input(df_clean, feature_columns=feature_cols)

    scaler = StandardScaler()
    X = scaler.fit_transform(X_raw)
    return X, feature_cols, df_clean


# ════════════════════════════════════════════════════════════════════════
# EMBEDDING-SPACE CLUSTER QUALITY
# ════════════════════════════════════════════════════════════════════════


def evaluate_embedding_silhouette(
    embedding: np.ndarray,
    n_clusters: int = DEFAULT_N_CLUSTERS,
    random_state: int = RANDOM_STATE,
) -> float:
    """Fit KMeans in the embedding space and return the silhouette score.

    This is the standard "does the reducer preserve structure?" probe used
    across all five technique files. Returns -1.0 when only one cluster is
    found (e.g. collapsed embedding).
    """
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=5)
    labels = km.fit_predict(embedding)
    if len(set(labels)) < 2:
        return -1.0
    return float(silhouette_score(embedding, labels))


# ════════════════════════════════════════════════════════════════════════
# SUBSAMPLING — used by KPCA / t-SNE / UMAP / Isomap for kernel-cost paths
# ════════════════════════════════════════════════════════════════════════


def subsample_indices(
    n_samples: int, n_target: int, random_state: int = RANDOM_STATE
) -> np.ndarray:
    """Deterministic subsample indices for expensive O(n^2) methods."""
    rng = np.random.default_rng(random_state)
    return rng.choice(n_samples, min(n_target, n_samples), replace=False)




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 3.1: Principal Component Analysis via SVD
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Derive PCA from the Singular Value Decomposition (X = U S V^T)
#   - Read a scree plot and pick n_components by variance threshold
#   - Interpret loadings to assign business meaning to each component
#   - Quantify compression quality via reconstruction error
#   - Recognise when PCA is the right tool (linear, fast, invertible)
#
# PREREQUISITES: MLFP04 Exercise 1 (clustering) + linear algebra basics.
#
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory — why PCA = SVD of the centred data matrix
#   2. Build — compute SVD, verify against sklearn.PCA
#   3. Train — scree plot + three component-selection criteria
#   4. Visualise — loadings heatmap + reconstruction error curve
#   5. Apply — Shopee Singapore customer analytics compression
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from sklearn.decomposition import PCA

from kailash_ml import ModelVisualizer



## THEORY — PCA is SVD


In [ ]:
# Given a centred (and usually standardised) data matrix X, SVD gives:
#     X = U  S  V^T
# Columns of V are the principal directions, singular values s_k encode
# strength, and variance explained by PC_k is s_k^2 / (n - 1).



## TASK 2 — BUILD: compute SVD, derive explained variance


In [ ]:
X, feature_cols, _ = load_customer_matrix()
n_samples, n_features = X.shape
print(f"=== E-commerce customers ===")
print(f"Samples: {n_samples:,}  Features: {n_features}")

# TODO: compute the SVD of X. Use np.linalg.svd with full_matrices=False.
# Hint: U, S, Vt = np.linalg.svd(...)
U, S, Vt = ____

# TODO: compute explained variance from singular values.
# Hint: variance of PC_k = S[k]^2 / (n_samples - 1)
explained_variance = ____
total_variance = explained_variance.sum()
evr = explained_variance / total_variance
# TODO: compute the cumulative explained variance ratio.
# Hint: np.cumsum
cum_evr = ____

print(f"\nTotal variance (~n_features={n_features}): {total_variance:.2f}")
print(f"\nTop 10 principal components:")
print(f"{'PC':>4} {'Sing. val':>12} {'Expl. var %':>14} {'Cumulative %':>14}")
print("-" * 48)
for i in range(min(10, n_features)):
    print(f"{i + 1:>4} {S[i]:>12.4f} {evr[i]:>13.2%} {cum_evr[i]:>13.2%}")

# Cross-check against sklearn's implementation.
pca_full = PCA(n_components=n_features)
pca_full.fit(X)
max_diff = float(np.abs(pca_full.explained_variance_ratio_ - evr).max())
print(f"\nSVD vs sklearn PCA max diff: {max_diff:.2e}")



### Checkpoint 1


In [ ]:
assert max_diff < 1e-6, f"SVD must match sklearn.PCA, got {max_diff:.2e}"
assert abs(cum_evr[-1] - 1.0) < 1e-6, "cumulative variance must sum to 1"
print("[ok] Checkpoint 1 — SVD-derived PCA matches sklearn\n")



## TASK 3 — TRAIN: scree plot + three selection criteria


In [ ]:
# TODO: find the number of components needed for 80 / 90 / 95% variance.
# Hint: np.searchsorted(cum_evr, 0.90) + 1 gives the first k that crosses.
n_80 = ____
n_90 = ____
n_95 = ____

# Kaiser: retain components whose eigenvalue > 1.
# TODO: count eigenvalues (== explained_variance) greater than 1.
n_kaiser = ____

# Broken-stick reference (already wired up).
broken_stick = np.array(
    [
        sum(1.0 / j for j in range(i, n_features + 1)) / n_features
        for i in range(1, n_features + 1)
    ]
)
n_broken = int((evr > broken_stick).sum())

print(f"=== Component-selection criteria ===")
print(f"  80% variance threshold : {n_80} components")
print(f"  90% variance threshold : {n_90} components")
print(f"  95% variance threshold : {n_95} components")
print(f"  Kaiser (eigenvalue>1)  : {n_kaiser} components")
print(f"  Broken-stick           : {n_broken} components")
print(
    f"  Compression at 95%     : "
    f"{n_features} -> {n_95} ({n_features / max(n_95, 1):.1f}x)"
)



### Checkpoint 2


In [ ]:
assert n_80 <= n_90 <= n_95 <= n_features, "monotonic thresholds"
assert 1 <= n_kaiser <= n_features
print("[ok] Checkpoint 2 — variance thresholds + Kaiser + broken-stick\n")



## TASK 4 — VISUALISE: scree + loadings + reconstruction error


In [ ]:
viz = ModelVisualizer()

# (a) Scree plot
fig_scree = viz.training_history(
    {
        "Explained variance %": (evr[:20] * 100).tolist(),
        "Cumulative %": (cum_evr[:20] * 100).tolist(),
    },
    x_label="Principal component",
)
fig_scree.update_layout(title="PCA scree: explained variance by component")
scree_path = OUTPUT_DIR / "01_pca_scree.html"
fig_scree.write_html(str(scree_path))
print(f"Saved: {scree_path}")

# (b) Loadings — which features drive each PC
n_pcs_inspect = min(5, n_features)
# TODO: build the loadings matrix from Vt. Loadings for the first K PCs
# are the first K rows of Vt transposed to (n_features, K).
loadings = ____

print(f"\n=== Loadings on first {n_pcs_inspect} PCs ===")
for i in range(n_pcs_inspect):
    col_norm = float(np.linalg.norm(loadings[:, i]))
    assert abs(col_norm - 1.0) < 1e-6, f"PC{i + 1} must be unit norm"
    abs_l = np.abs(loadings[:, i])
    top = np.argsort(abs_l)[::-1][:3]
    names = [f"{feature_cols[j]} ({loadings[j, i]:+.2f})" for j in top]
    print(f"  PC{i + 1}: {', '.join(names)}")

# (c) Reconstruction error as a function of k
n_range = list(range(1, min(n_features + 1, 21)))
recon_errors = []
for k in n_range:
    # TODO: fit a k-component PCA, project, then inverse_transform and
    # compute the mean squared error between X and the reconstruction.
    # Hint: pca_k.fit_transform + pca_k.inverse_transform
    pca_k = PCA(n_components=k, random_state=42)
    X_proj = ____
    X_hat = ____
    recon_errors.append(float(np.mean((X - X_hat) ** 2)))

fig_recon = viz.training_history(
    {"Reconstruction MSE": recon_errors}, x_label="Components retained"
)
fig_recon.update_layout(title="PCA: reconstruction error vs components")
recon_path = OUTPUT_DIR / "01_pca_reconstruction.html"
fig_recon.write_html(str(recon_path))
print(f"Saved: {recon_path}")



### Checkpoint 3


In [ ]:
assert recon_errors[0] > recon_errors[-1], "MSE must fall as k grows"
assert recon_errors[-1] < 0.1, "full-rank reconstruction should be ~0"
print("\n[ok] Checkpoint 3 — visualisations + loadings + reconstruction\n")



## TASK 5 — APPLY: Shopee Singapore customer analytics compression


In [ ]:
# SCENARIO: Shopee SG segments ~14M shoppers every night from 120+
# behavioural features. K-means at full rank misses the 6-hour pre-dawn
# window. PCA compresses shoppers to ~18D, K-means finishes in ~42 min,
# and the homepage carousel refreshes on time. Freshness is worth
# ~S$180K/day in incremental GMV vs a tiny compute bill.

compression_ratio = n_features / max(n_95, 1)
print(f"=== Shopee-style compression estimate ===")
print(f"  Raw dimensions       : {n_features}")
print(f"  At 95% variance      : {n_95}")
print(f"  Compression ratio    : {compression_ratio:.1f}x")
print(f"  Variance discarded   : {(1 - cum_evr[n_95 - 1]) * 100:.2f}%")
print(
    f"  Reconstruction MSE   : {recon_errors[min(n_95 - 1, len(recon_errors) - 1)]:.4f}"
)



## REFLECTION


[x] Derived PCA directly from X = U S V^T and verified against sklearn
  [x] Built a scree plot and applied three selection criteria
  [x] Read loadings to assign business meaning to principal components
  [x] Measured reconstruction error as a compression quality metric
  [x] Costed PCA as the right tool for nightly Shopee segmentation

  KEY INSIGHT: PCA is the only linear dim-reduction method with a true
  inverse transform. If your downstream job needs to EXPLAIN a customer
  in the original feature space, PCA is the answer.

  Next: 02_kernel_pca.py lifts this into nonlinear territory.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

